# Notebook 2 — Safety Metrics Evaluation

This notebook applies the full safety metrics framework to real Waymo data:
- Time-to-Collision (TTC)
- DRAC (Deceleration Rate to Avoid Crash)
- Proximity scores
- Jerk and comfort metrics
- Per-segment evaluation report

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.waymo_loader import load_dataset, extract_trajectories
from src.metrics.safety_metrics import batch_ttc, comfort_metrics, nearest_agent_distances, speed_deviation
from src.metrics.evaluation_framework import AVEvaluationFramework, EvaluationConfig
from src.visualization.plots import plot_ttc_distribution, plot_safety_heatmap, plot_speed_profile

sns.set_theme(style='whitegrid')

In [2]:
dataset = load_dataset('../data', max_segments=5)
trajectories = extract_trajectories(dataset['lidar_box'])
print(f'Loaded {len(trajectories):,} trajectory rows across {trajectories["segment_id"].nunique()} segments')

Loaded 62,312 trajectory rows across 5 segments


## Time-to-Collision Analysis

In [3]:
print('Computing TTC for all frames...')
ttc_df = batch_ttc(trajectories)
print(f'Frames with finite TTC: {ttc_df["min_ttc"].replace(np.inf, np.nan).notna().sum():,}')
print(f'Near-miss events (TTC < 3s): {ttc_df["near_miss"].sum():,}')
ttc_df.head()

Computing TTC for all frames...
Frames with finite TTC: 986
Near-miss events (TTC < 3s): 710


/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)


,segment_id,timestamp_us,min_ttc,near_miss
0,10203656353524179475_7625_000_7645_000,1522688015070129,0.549877,True
1,10203656353524179475_7625_000_7645_000,1522688015170074,0.440067,True
2,10203656353524179475_7625_000_7645_000,1522688015269987,0.377552,True
3,10203656353524179475_7625_000_7645_000,1522688015369887,0.358990,True
4,10203656353524179475_7625_000_7645_000,1522688015469787,0.380104,True


In [4]:
fig = plot_ttc_distribution(ttc_df['min_ttc'])
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_26566/2967755119.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Comfort Metrics

In [5]:
comfort = comfort_metrics(trajectories)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
comfort['max_jerk'].clip(0, 20).hist(bins=40, ax=axes[0], color='mediumpurple', edgecolor='white')
axes[0].set_title('Max Jerk per Track (m/s³)')
axes[0].axvline(6, color='red', linestyle='--', label='Threshold')
axes[0].legend()

comfort['max_acceleration'].clip(0, 10).hist(bins=40, ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Max Acceleration per Track (m/s²)')
axes[1].axvline(4, color='red', linestyle='--', label='Threshold')
axes[1].legend()

comfort['mean_speed'].clip(0, 30).hist(bins=40, ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Mean Speed per Track (m/s)')

plt.tight_layout()
plt.show()

/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(total_distance)
/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_26566/2552914554.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Full Evaluation Framework

In [6]:
config = EvaluationConfig(
    near_miss_ttc_threshold_s=3.0,
    speed_limit_mps=15.0,
    max_acceptable_jerk_mps3=6.0,
)
framework = AVEvaluationFramework(config)
dataset_report = framework.evaluate_dataset(trajectories)
summary = dataset_report.summary_df()
print(f'Evaluated {dataset_report.n_segments} segments')
summary

/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(total_distance)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/

/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(total_distance)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)


/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(total_distance)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/

/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: invalid value encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)
/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:115: RuntimeWarning: divide by zero encountered in divide
  ttc = np.where(closing > 0, gap / closing, np.inf)


Evaluated 5 segments


/Users/sachiniweerasekara/Desktop/Waymo_Project/av-safety-evaluation/notebooks/../src/metrics/safety_metrics.py:230: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(total_distance)


,segment_id,n_frames,n_agents,min_ttc_overall,mean_min_ttc,near_miss_count,near_miss_rate_pct,mean_jerk,max_jerk,jerk_violation_rate_pct,mean_speed_mps,speed_excess_rate_pct,mean_nearest_dist_m,min_nearest_dist_m,proximity_warning_rate_pct
0,10203656353524179475_7625_000_7645_000,197,136,0.000000,0.248157,197,100.000000,0.099617,6.640603,0.098122,2.518449,12.489487,5.861262,0.001474,61.536305
1,1024360143612057520_3580_000_3600_000,198,143,0.000000,0.015547,198,100.000000,0.067632,14.221341,0.011175,0.473160,0.000000,3.763168,0.028780,76.169190
2,10247954040621004675_2180_000_2200_000,197,51,0.000000,84.449636,21,10.659898,0.021661,1.074659,0.000000,0.024871,0.000000,5.898403,0.745555,32.280493
3,10289507859301986274_4200_000_4220_000,197,245,0.000000,0.286277,197,100.000000,0.133022,24.010642,0.031520,0.935013,0.000000,3.655120,0.000645,74.125311
4,10335539493577748957_1372_870_1392_870,197,63,0.098803,39.150316,97,49.238579,0.097903,2.134607,0.000000,5.247615,22.739578,9.391368,0.003227,42.014077


In [7]:
fig = plot_safety_heatmap(dataset_report)
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_26566/3182921107.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Speed Deviation from Limit

In [8]:
vehicles = trajectories[trajectories['object_type_name'] == 'vehicle']
seg_id = vehicles['segment_id'].iloc[0]
fig = plot_speed_profile(vehicles, segment_id=seg_id)
plt.show()

/var/folders/jj/khf1rsjs76137181nr0hly600000gn/T/ipykernel_26566/135741941.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
